# PYNQ 上板 LeNet-5 CNN 推理测试

本 notebook 在 PYNQ-Z2 上运行，验证 CIM SoC 能否通过 im2col 正确运行 LeNet-5 CNN。

**网络结构：**
Conv1(1→6, 5×5) → Pool(2×2) → Conv2(6→16, 5×5) → Pool(2×2) → FC3(256→120) → FC4(120→84) → FC5(84→10)

**硬件映射：**
- Conv 层：Python 端 im2col 展开 → 硬件 MVM（每个 output pixel 一次）
- Pool 层：Python 端 max pooling（不需要硬件）
- FC 层：硬件 MVM 直接计算

**前提：**
1. 宿主机已运行 `lenet5_quantize.py`，生成 `lenet5_data.zip`
2. 已上传 `lenet5_data.zip`、`cim_soc.bit`、`cim_soc.hwh` 到 Jupyter 同目录

In [1]:
!unzip -o lenet5_data.zip

Archive:  lenet5_data.zip
   creating: lenet5_data/
  inflating: lenet5_data/fc4_weight_tiles.hex  
  inflating: lenet5_data/conv1_weight_tiles.hex  
  inflating: lenet5_data/quant_params.hex  
  inflating: lenet5_data/conv2_weight_tiles.hex  
  inflating: lenet5_data/fc5_bias.hex  
  inflating: lenet5_data/zero_points.hex  
  inflating: lenet5_data/fc3_bias.hex  
  inflating: lenet5_data/fc4_bias.hex  
   creating: lenet5_data/test_images/
 extracting: lenet5_data/test_images/img_0004_label.txt  
  inflating: lenet5_data/test_images/img_0006_fc5.hex  
 extracting: lenet5_data/test_images/img_0008_pred.txt  
 extracting: lenet5_data/test_images/img_0018_pred.txt  
 extracting: lenet5_data/test_images/img_0012_label.txt  
  inflating: lenet5_data/test_images/img_0007_fc5.hex  
 extracting: lenet5_data/test_images/img_0009_pred.txt  
 extracting: lenet5_data/test_images/img_0003_pred.txt  
  inflating: lenet5_data/test_images/img_0013.hex  
 extracting: lenet5_data/test_images/img_0015_p

In [2]:
from pynq import Overlay, MMIO
import numpy as np
import os, glob, time

## 1. 加载 Bitstream

In [3]:
ol = Overlay('cim_soc.bit')
print('Overlay loaded!')

BASE = 0x40000000
mmio = MMIO(BASE, 0x4000)

Overlay loaded!


## 2. CSR 地址 + 硬件操作函数

In [4]:
CTRL          = 0x000
STATUS        = 0x004
CSR_IN_DIM    = 0x010
CSR_OUT_DIM   = 0x014
CSR_N_IB      = 0x018
CSR_N_OB      = 0x01C
REQUANT_MULT  = 0x020
REQUANT_SHIFT = 0x024
INPUT_ZP      = 0x028
ACT_MODE      = 0x02C
CYCLE_CNT_LO  = 0x030
MAC_CNT_LO    = 0x038
PRED_CLASS    = 0x040
WDMA_ADDR     = 0x044
WDMA_DATA     = 0x048
WDMA_CTRL     = 0x04C
LOGIT_BASE    = 0x100
MEM_INPUT     = 0x1000
MEM_BIAS      = 0x2000

TILE_ROWS = 16
TILE_COLS = 16
ELEMS_PER_CHUNK = 4
CHUNKS_PER_ROW = 4
CHUNKS_PER_TILE = 64

In [5]:
def soft_reset():
    mmio.write(CTRL, 0x4)

def clear_done():
    mmio.write(CTRL, 0x2)

def configure_layer(in_dim, out_dim, zp, mult, shift, relu):
    n_ib = (in_dim + 15) // 16
    n_ob = (out_dim + 15) // 16
    mmio.write(CSR_IN_DIM, in_dim)
    mmio.write(CSR_OUT_DIM, out_dim)
    mmio.write(CSR_N_IB, n_ib)
    mmio.write(CSR_N_OB, n_ob)
    mmio.write(REQUANT_MULT, mult)
    mmio.write(REQUANT_SHIFT, shift)
    mmio.write(INPUT_ZP, int(zp) & 0xFFFFFFFF)
    mmio.write(ACT_MODE, 1 if relu else 0)

def load_weights_burst(chunks):
    mmio.write(WDMA_ADDR, 0)
    mmio.write(WDMA_CTRL, 0x02)
    for c in chunks:
        mmio.write(WDMA_DATA, int(c))
    mmio.write(WDMA_CTRL, 0x00)

def load_bias(bias_list):
    for i, b in enumerate(bias_list):
        mmio.write(MEM_BIAS + 4*i, int(b) & 0xFFFFFFFF)

def load_input(data_u8):
    padded = list(data_u8)
    while len(padded) % 16 != 0:
        padded.append(0)
    for i, x in enumerate(padded):
        mmio.write(MEM_INPUT + 4*i, int(x) & 0xFF)

def run_inference():
    clear_done()
    mmio.write(CTRL, 0x1)
    while not (mmio.read(STATUS) & 0x2):
        pass
    return mmio.read(CYCLE_CNT_LO), mmio.read(MAC_CNT_LO)

def read_output(out_dim):
    out = []
    for i in range(out_dim):
        v = mmio.read(LOGIT_BASE + 4*i)
        out.append(np.uint8(v & 0xFF).view(np.int8))
    return np.array(out, dtype=np.int8)

def hw_mvm(x_u8, w_chunks, bias_u32, out_dim, zp, mult, shift, relu):
    """Run one MVM on hardware, return INT8 output array."""
    in_dim = len(x_u8)
    soft_reset()
    configure_layer(in_dim, out_dim, zp, mult, shift, relu)
    load_weights_burst(w_chunks)
    load_bias(bias_u32)
    load_input(x_u8)
    cycles, macs = run_inference()
    return read_output(out_dim), cycles

print('HW functions loaded.')

HW functions loaded.


## 3. im2col + MaxPool（Python 端）

In [6]:
def im2col(feat, kh, kw, stride=1, padding=0):
    """Explicit im2col: [C,H,W] UINT8 → [C*kh*kw, oh*ow] UINT8."""
    C, H, W = feat.shape
    if padding > 0:
        p = np.zeros((C, H+2*padding, W+2*padding), dtype=feat.dtype)
        p[:, padding:padding+H, padding:padding+W] = feat
        feat = p
    Hp, Wp = feat.shape[1], feat.shape[2]
    oh = (Hp - kh) // stride + 1
    ow = (Wp - kw) // stride + 1
    col = np.zeros((C*kh*kw, oh*ow), dtype=feat.dtype)
    idx = 0
    for i in range(oh):
        for j in range(ow):
            col[:, idx] = feat[:, i*stride:i*stride+kh, j*stride:j*stride+kw].flatten()
            idx += 1
    return col, oh, ow

def maxpool2d(feat, k=2, s=2):
    """Max pooling on signed INT8 feature map [C,H,W]."""
    C, H, W = feat.shape
    oh, ow = H // s, W // s
    out = np.zeros((C, oh, ow), dtype=feat.dtype)
    for c in range(C):
        for i in range(oh):
            for j in range(ow):
                out[c, i, j] = feat[c, i*s:i*s+k, j*s:j*s+k].max()
    return out

def weight_to_chunks(w_i8):
    """Pack [out_dim, in_dim] INT8 → list of 32-bit chunk words."""
    od, id_ = w_i8.shape
    n_ob = (od + TILE_ROWS - 1) // TILE_ROWS
    n_ib = (id_ + TILE_COLS - 1) // TILE_COLS
    chunks = []
    for ob in range(n_ob):
        for ib in range(n_ib):
            for chunk in range(CHUNKS_PER_TILE):
                row = chunk // CHUNKS_PER_ROW
                cg = chunk % CHUNKS_PER_ROW
                word = 0
                for b in range(ELEMS_PER_CHUNK):
                    oi = ob * TILE_ROWS + row
                    ii = ib * TILE_COLS + cg * ELEMS_PER_CHUNK + b
                    if oi < od and ii < id_:
                        word |= (int(w_i8[oi, ii]) & 0xFF) << (b * 8)
                chunks.append(word)
    return chunks

def bias_to_u32(b):
    return [int(v) & 0xFFFFFFFF for v in b]

print('im2col / pool / packing functions loaded.')

im2col / pool / packing functions loaded.


## 4. 读取 Hex + 加载层参数

In [7]:
DATA_DIR = 'lenet5_data'

def read_hex_u32(path):
    with open(path) as f:
        return [int(l.strip(), 16) for l in f if l.strip()]

def read_hex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

def read_hex_int8(path):
    return np.array([np.uint8(v).view(np.int8) for v in read_hex_u8(path)], dtype=np.int8)

# Read layer info
with open(f'{DATA_DIR}/layer_info.txt') as f:
    print(f.read())

conv1: conv 1ch 5x5 → 6ch stride=1 pad=0 mult=70 shift=16
pool1: maxpool 2x2 stride=2
conv2: conv 6ch 5x5 → 16ch stride=1 pad=0 mult=98 shift=16
pool2: maxpool 2x2 stride=2
fc3: fc 256→120 mult=118 shift=16 relu=True
fc4: fc 120→84 mult=227 shift=16 relu=True
fc5: fc 84→10 mult=163 shift=16 relu=False



In [8]:
# Load quant params: pairs of (mult, shift) for each compute layer
qp = read_hex_u32(f'{DATA_DIR}/quant_params.hex')
zps = read_hex_u32(f'{DATA_DIR}/zero_points.hex')

# conv1, conv2, fc3, fc4, fc5 — 5 compute layers
layer_names = ['conv1', 'conv2', 'fc3', 'fc4', 'fc5']
layers = {}
for i, name in enumerate(layer_names):
    w_chunks = read_hex_u32(f'{DATA_DIR}/{name}_weight_tiles.hex')
    bias_u32 = read_hex_u32(f'{DATA_DIR}/{name}_bias.hex')
    mult = qp[i * 2]
    shift = qp[i * 2 + 1]
    zp = int(np.int32(np.uint32(zps[i])))
    layers[name] = {
        'w_chunks': w_chunks,
        'bias_u32': bias_u32,
        'mult': mult,
        'shift': shift,
        'zp': zp,
    }
    print(f'  {name}: {len(w_chunks)} w_chunks, {len(bias_u32)} bias, mult={mult}, shift={shift}, zp={zp}')

  conv1: 128 w_chunks, 6 bias, mult=70, shift=16, zp=0
  conv2: 640 w_chunks, 16 bias, mult=98, shift=16, zp=0
  fc3: 8192 w_chunks, 120 bias, mult=118, shift=16, zp=0
  fc4: 3072 w_chunks, 84 bias, mult=227, shift=16, zp=0
  fc5: 384 w_chunks, 10 bias, mult=163, shift=16, zp=0


In [9]:
# Conv layer metadata (must match lenet5_quantize.py)
CONV1 = {'C_out': 6,  'C_in': 1, 'K': 5, 'stride': 1, 'pad': 0, 'relu': True}
CONV2 = {'C_out': 16, 'C_in': 6, 'K': 5, 'stride': 1, 'pad': 0, 'relu': True}
FC3   = {'in_dim': 256, 'out_dim': 120, 'relu': True}
FC4   = {'in_dim': 120, 'out_dim': 84,  'relu': True}
FC5   = {'in_dim': 84,  'out_dim': 10,  'relu': False}

## 5. AXI 连通性测试

In [10]:
soft_reset()
mmio.write(CSR_IN_DIM, 784)
rb = mmio.read(CSR_IN_DIM)
print(f'IN_DIM write=784, readback={rb}  {"PASS" if rb == 784 else "FAIL"}')
assert rb == 784, 'AXI failed!'

IN_DIM write=784, readback=784  PASS


## 6. LeNet-5 硬件推理函数

Conv 层：im2col 展开后逐 pixel 调用硬件 MVM
Pool 层：Python 端 max pooling
FC 层：直接调用硬件 MVM

In [11]:
def hw_conv_layer(feat_u8, conv_cfg, layer_params):
    """
    Conv layer via im2col + per-pixel HW MVM.
    feat_u8: [C_in, H, W] UINT8
    Returns: [C_out, oh, ow] INT8
    """
    C_out = conv_cfg['C_out']
    K = conv_cfg['K']
    col, oh, ow = im2col(feat_u8, K, K, conv_cfg['stride'], conv_cfg['pad'])
    n_pix = oh * ow
    col_len = col.shape[0]
    
    # Pack weight once: [C_out, col_len] → chunks
    # (w_chunks already loaded from hex, pre-packed)
    out_flat = np.zeros((C_out, n_pix), dtype=np.int8)
    total_cycles = 0
    
    for p in range(n_pix):
        col_vec = col[:, p].tolist()
        out_p, cyc = hw_mvm(
            col_vec, layer_params['w_chunks'], layer_params['bias_u32'],
            C_out, layer_params['zp'], layer_params['mult'],
            layer_params['shift'], conv_cfg['relu'])
        out_flat[:, p] = out_p
        total_cycles += cyc
    
    return out_flat.reshape(C_out, oh, ow), total_cycles

def hw_fc_layer(x_u8, fc_cfg, layer_params):
    """FC layer via single HW MVM."""
    out, cyc = hw_mvm(
        x_u8, layer_params['w_chunks'], layer_params['bias_u32'],
        fc_cfg['out_dim'], layer_params['zp'], layer_params['mult'],
        layer_params['shift'], fc_cfg['relu'])
    return out, cyc

def hw_infer_lenet5(img_u8_flat):
    """
    Full LeNet-5 inference on hardware.
    img_u8_flat: [784] UINT8
    Returns: (pred, total_cycles, per_layer_cycles)
    """
    x = img_u8_flat.reshape(1, 28, 28)  # [C=1, H=28, W=28]
    timing = {}
    
    # Conv1: [1,28,28] → [6,24,24]
    x_out, cyc = hw_conv_layer(x, CONV1, layers['conv1'])
    timing['conv1'] = cyc
    
    # Pool1: [6,24,24] → [6,12,12]
    x_pool = maxpool2d(x_out.view(np.int8), 2, 2).view(np.uint8)
    
    # Conv2: [6,12,12] → [16,8,8]
    x_out2, cyc = hw_conv_layer(x_pool, CONV2, layers['conv2'])
    timing['conv2'] = cyc
    
    # Pool2: [16,8,8] → [16,4,4]
    x_pool2 = maxpool2d(x_out2.view(np.int8), 2, 2).view(np.uint8)
    
    # Flatten: [16,4,4] → [256]
    x_flat = x_pool2.flatten()
    
    # FC3: 256 → 120
    # reinterpret signed→unsigned
    if x_flat.dtype == np.int8:
        x_flat = x_flat.view(np.uint8)
    fc3_out, cyc = hw_fc_layer(x_flat, FC3, layers['fc3'])
    timing['fc3'] = cyc
    
    # FC4: 120 → 84
    fc4_in = np.array(fc3_out, dtype=np.int8).view(np.uint8)
    fc4_out, cyc = hw_fc_layer(fc4_in, FC4, layers['fc4'])
    timing['fc4'] = cyc
    
    # FC5: 84 → 10
    fc5_in = np.array(fc4_out, dtype=np.int8).view(np.uint8)
    fc5_out, cyc = hw_fc_layer(fc5_in, FC5, layers['fc5'])
    timing['fc5'] = cyc
    
    pred = int(np.argmax(fc5_out))
    total_cyc = sum(timing.values())
    return pred, fc5_out, total_cyc, timing

print('LeNet-5 HW inference function ready.')

LeNet-5 HW inference function ready.


## 7. 逐张推理测试

对每张图：硬件跑完整 LeNet-5 → 对比 Python golden 预测

In [12]:
img_dir = f'{DATA_DIR}/test_images'
img_files = sorted(glob.glob(f'{img_dir}/img_????.hex'))
n_images = len(img_files)
print(f'Found {n_images} test images\n')

hw_correct = 0
bit_exact = 0
total = 0
error_imgs = []
all_timing = []

t_start = time.time()

for img_path in img_files:
    name = os.path.basename(img_path).replace('.hex', '')
    
    # Read data
    img_u8 = np.array(read_hex_u8(img_path), dtype=np.uint8)
    label = int(open(f'{img_dir}/{name}_label.txt').read().strip())
    py_pred = int(open(f'{img_dir}/{name}_pred.txt').read().strip())
    fc5_golden = read_hex_int8(f'{img_dir}/{name}_fc5.hex')
    
    # HW inference
    hw_pred, fc5_hw, total_cyc, timing = hw_infer_lenet5(img_u8)
    all_timing.append(timing)
    
    # Compare
    fc5_ok = np.array_equal(fc5_hw, fc5_golden)
    pred_ok = (hw_pred == py_pred)
    all_ok = fc5_ok and pred_ok
    
    if hw_pred == label:
        hw_correct += 1
    if all_ok:
        bit_exact += 1
    else:
        error_imgs.append(name)
    total += 1
    
    mark = '\u2713' if all_ok else '\u2717'
    correct_mark = '\u2713' if hw_pred == label else '\u2717'
    print(
        f'  {name}: label={label} hw={hw_pred} py={py_pred} '
        f'fc5={"OK" if fc5_ok else "ERR"} '
        f'exact={mark} correct={correct_mark} '
        f'({total_cyc} cycles)'
    )
    
    if not fc5_ok:
        for j in range(10):
            if fc5_hw[j] != fc5_golden[j]:
                print(f'    FC5[{j}]: hw={fc5_hw[j]}, golden={fc5_golden[j]}')

t_elapsed = time.time() - t_start
print(f'\nTotal wall time: {t_elapsed:.1f}s ({t_elapsed/total:.2f}s per image)')

Found 20 test images

  img_0000: label=7 hw=7 py=7 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0001: label=2 hw=2 py=2 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0002: label=1 hw=1 py=1 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0003: label=0 hw=0 py=0 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0004: label=4 hw=4 py=4 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0005: label=1 hw=1 py=1 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0006: label=4 hw=4 py=4 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0007: label=9 hw=9 py=9 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0008: label=5 hw=5 py=5 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0009: label=9 hw=9 py=9 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0010: label=0 hw=0 py=0 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0011: label=6 hw=6 py=6 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0012: label=9 hw=9 py=9 fc5=OK exact=✓ correct=✓ (76034 cycles)
  img_0013: label=0 hw=0 py=0 fc5=OK exact=✓ correct=✓ (76034 cycles

## 8. 结果总结

In [13]:
print('=' * 60)
print('LeNet-5 CNN Real Inference Results')
print('=' * 60)
print(f'  Total images:          {total}')
print(f'  HW accuracy:           {100.*hw_correct/total:.1f}% ({hw_correct}/{total})')
print(f'  Bit-exact (vs Python): {100.*bit_exact/total:.1f}% ({bit_exact}/{total})')
print(f'  Computation errors:    {len(error_imgs)}')
print()

# Per-layer average cycles
if all_timing:
    avg = {}
    for key in all_timing[0]:
        avg[key] = int(np.mean([t[key] for t in all_timing]))
    print('  Per-layer avg cycles:')
    for k, v in avg.items():
        print(f'    {k:8s}: {v:>8d} cycles')
    print(f'    {"total":8s}: {sum(avg.values()):>8d} cycles')
    print()

if len(error_imgs) == 0:
    print('>>> ALL IMAGES BIT-EXACT MATCH <<<')
    print('>>> CIM SoC correctly runs LeNet-5 CNN on real MNIST! <<<')
else:
    print(f'  Mismatched images: {error_imgs}')
print('=' * 60)

LeNet-5 CNN Real Inference Results
  Total images:          20
  HW accuracy:           95.0% (19/20)
  Bit-exact (vs Python): 100.0% (20/20)
  Computation errors:    0

  Per-layer avg cycles:
    conv1   :    63360 cycles
    conv2   :    10112 cycles
    fc3     :     1552 cycles
    fc4     :      876 cycles
    fc5     :      134 cycles
    total   :    76034 cycles

>>> ALL IMAGES BIT-EXACT MATCH <<<
>>> CIM SoC correctly runs LeNet-5 CNN on real MNIST! <<<
